<a href="https://colab.research.google.com/github/AmitoVrito/Traceprop/blob/main/notebooks/runs/notebooks/pilot_p160_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Traceprop inline-precond — Pythia-160M / SST-2 pilot

Run cells 1-6 top to bottom. Cell 4 is the 30-subset **pilot**; cell 5 is the gate that
decides whether to launch the full 200-subset x 5-seed campaign. Cell 7 (Part B, Pythia-1B
overhead) is separate — run it on its own only when you choose to.

Clones `main` (latest) during HF bring-up.


## Cell 1 — GPU check + mount Drive

In [1]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE")
assert any(g in gpu for g in ("A100", "L4")), (
    f"Need an A100 or L4 GPU, got {gpu!r}. "
    "Runtime -> Change runtime type -> GPU (A100 or L4).")

from google.colab import drive
drive.mount('/content/drive')

GPU: NVIDIA A100-SXM4-40GB
Mounted at /content/drive


## Cell 2 — Clone repo at the pinned commit + install deps

In [2]:
%cd /content
!rm -rf Traceprop && git clone -q https://github.com/AmitoVrito/Traceprop.git
%cd /content/Traceprop
!git checkout -q main
!git log --oneline -1
import subprocess as _sp
_fix = _sp.run(["grep","-q","_tok.pad_token_id","experiments/exp27_lds_quality.py"]).returncode == 0
print("PAD-TOKEN FIX PRESENT:", _fix)
assert _fix, "STALE CLONE missing the pad_token_id fix -- re-open the notebook fresh from GitHub"

# Colab ships torch (CUDA), transformers, scipy, numpy. Add the rest.
!pip -q install "datasets>=2.14" "peft>=0.7" pytest
!pip -q install --ignore-requires-python logix-ai   # setup.py caps py<3.11; wheel is pure-python
!pip -q uninstall -y torchao   # Colab's torchao 0.10 breaks peft 0.20 LoRA dispatch; unused here
!pip -q install -e . --no-deps                        # traceprop, deps already present

import torch, transformers, peft, logix, scipy, numpy, datasets
print(f"torch {torch.__version__} | transformers {transformers.__version__} | "
      f"peft {peft.__version__} | logix {logix.__version__} | "
      f"datasets {datasets.__version__} | numpy {numpy.__version__} | scipy {scipy.__version__}")
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

/content
/content/Traceprop
23b539a (HEAD -> main, origin/main, origin/HEAD) Add noise-ceiling check (retrain N subsets x2, mean per-example Spearman) + wire into pilot
PAD-TOKEN FIX PRESENT: True
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 5.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for traceprop (pyproject.toml) ... done
torch 2.11.0+cu128 | transformers 5.16.1 | peft 0.20.0 | logix 0.1.1 | datasets 4.8.5 | numpy 2.1.3 | scipy 1.16.3
CUDA: True NVIDIA A100-SXM4-40GB


## Cell 3 — Unit tests (stops if anything fails)

In [3]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest",
                    "tests/unit/test_lora_logging.py", "-q"])
assert r.returncode == 0, "UNIT TESTS FAILED - do not proceed."
print("unit tests passed")

unit tests passed


## Cell 3b — DIAGNOSE the HF classifier (fast; run before the pilot)

In [4]:
import subprocess, sys, os
os.chdir('/content/Traceprop/experiments')
# SST-2 learns at lr 1e-4/2e-4 but overfits on 500 ex (test ~0.72). Try n=2000.
for lr in ("1e-4", "2e-4"):
    print(f"\n############## sst2  n=2000  lr={lr}  epochs=10  seq=64 ##############")
    r = subprocess.run([sys.executable, "diag_hf_classifier.py",
                        "--model", "EleutherAI/pythia-160m", "--data", "sst2",
                        "--n", "2000", "--n_test", "300", "--epochs", "10", "--seq", "64",
                        "--lr", lr, "--device", "cuda"], capture_output=True, text=True)
    for ln in r.stdout.splitlines():
        if ln.startswith(("[train]", "[eval]", "[verdict]")):
            print(ln)
    if r.returncode != 0:
        print("ERR:", r.stderr[-600:])


############## sst2  n=2000  lr=1e-4  epochs=10  seq=64 ##############
[train] epoch 1/10  mean loss=0.7354
[train] epoch 2/10  mean loss=0.5526
[train] epoch 3/10  mean loss=0.4090
[train] epoch 4/10  mean loss=0.3034
[train] epoch 5/10  mean loss=0.2239
[train] epoch 6/10  mean loss=0.1604
[train] epoch 7/10  mean loss=0.1072
[train] epoch 8/10  mean loss=0.0652
[train] epoch 9/10  mean loss=0.0418
[train] epoch 10/10  mean loss=0.0238
[eval] test acc=0.7767  pred distribution (argmax)=[132, 168]  
[verdict] OK - model learns, signal viable

############## sst2  n=2000  lr=2e-4  epochs=10  seq=64 ##############
[train] epoch 1/10  mean loss=0.7400
[train] epoch 2/10  mean loss=0.5060
[train] epoch 3/10  mean loss=0.3970
[train] epoch 4/10  mean loss=0.2652
[train] epoch 5/10  mean loss=0.1915
[train] epoch 6/10  mean loss=0.1501
[train] epoch 7/10  mean loss=0.1165
[train] epoch 8/10  mean loss=0.0685
[train] epoch 9/10  mean loss=0.0637
[train] epoch 10/10  mean loss=0.0316
[eval] 

## Cell 4 — PILOT: Pythia-160M, SST-2, seed 0, 30 subsets, n_train 1500

In [5]:
import time, subprocess, sys, os, json

os.chdir('/content/Traceprop/experiments')
cmd = [sys.executable, "exp35_logix_lds.py",
       "--backend", "hf", "--model", "EleutherAI/pythia-160m", "--data", "sst2",
       "--device", "cuda", "--gpu_check", "", "--track", "0",
       "--factored", "--inline_precond", "--tune_logix", "--logix_permodule_damping",
       "--n_train", "2000", "--n_test", "100", "--n_subsets", "30",
       "--subset_frac", "0.5", "--epochs", "10", "--batch", "16", "--lr", "2e-4", "--seq", "64",
       "--proj_dim", "512", "--lora_init", "pca", "--seed", "0",
       "--precond_damping_grid", "1e-6,1e-5,1e-4,1e-3,1e-2,1e-1,1e0,1e1",
       "--precond_val_frac", "0.3", "--min_target_acc", "0.72", "--noise_ceiling", "10",
       "--out", "results/exp35_p160_sst2_pilot_seed0.json", "--force"]
t0 = time.time()
p = subprocess.run(cmd, capture_output=True, text=True)
mins = (time.time() - t0) / 60

# persist logs to Drive so a failure/disconnect never loses the traceback
os.makedirs("/content/drive/MyDrive/traceprop_results/", exist_ok=True)
open("/content/drive/MyDrive/traceprop_results/pilot_stderr.txt", "w").write(p.stderr)
open("/content/drive/MyDrive/traceprop_results/pilot_stdout.txt", "w").write(p.stdout)

if p.returncode != 0:
    print("EXIT", p.returncode, "- PILOT FAILED. STDERR tail (also saved to Drive/pilot_stderr.txt):\n")
    print("\n".join(p.stderr.splitlines()[-45:]))
    raise SystemExit("pilot run failed - paste the STDERR tail above")

print(f"pilot runtime: {mins:.1f} min")
print("\n".join(p.stdout.splitlines()[-15:]))

d = json.load(open("results/exp35_p160_sst2_pilot_seed0.json"))
print(f"\ntarget test acc: {d.get('target_test_acc')}")
keys = ["logix_dot", "logix_preconditioned", "logix_preconditioned_tuned",
        "traceprop_factored_kfac8_dot", "traceprop_factored_kfac8_inlineprecond",
        "traceprop_factored_kfac7_inlineprecond", "random"]
print(f"\n{'method':<44}{'full LDS':>18}{'held-out eval':>16}")
for k in keys:
    if k in d["lds"]:
        f = d["lds"][k]
        h = d.get("lds_heldout_eval", {}).get(k, {})
        print(f"{k:<44}{f['mean']:+.4f} +/- {f['std']:.4f}   {h.get('mean', float('nan')):+.4f}")


EXIT 1 - PILOT FAILED. STDERR tail (also saved to Drive/pilot_stderr.txt):


Loading weights: 100%|██████████| 147/147 [00:00<00:00, 6037.63it/s]
[transformers] GPTNeoXForSequenceClassification LOAD REPORT from: EleutherAI/pythia-160m
Key              | Status     | 
-----------------+------------+-
embed_out.weight | UNEXPECTED | 
score.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[exp35] ABORT: target test acc 0.7100 < --min_target_acc 0.72 (pred class distribution [29, 71]). The model is not learning the task -- fix the SETUP (data/epochs/lr/head) before any LDS run. See diag_hf_classifier.py.


SystemExit: pilot run failed - paste the STDERR tail above

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Cell 5 — GATE: proceed to the full campaign, or stop?

In [ ]:
import json
d = json.load(open("/content/Traceprop/experiments/results/exp35_p160_sst2_pilot_seed0.json"))
lds = d["lds"]
methods = {k: v["mean"] for k, v in lds.items() if k != "random"}
best = max(methods, key=methods.get)
rand = lds.get("random", {}).get("mean", 0.0)
print(f"random LDS      : {rand:+.4f}")
print(f"best method     : {best} = {methods[best]:+.4f}")
print(f"inline (kfac8)  : {lds.get('traceprop_factored_kfac8_inlineprecond',{}).get('mean',float('nan')):+.4f}")
print(f"LogIX tuned     : {lds.get('logix_preconditioned_tuned',{}).get('mean',float('nan')):+.4f}")
print()
if methods[best] >= 0.2:
    print("PROCEED: signal is clearly above noise. Launch the 200-subset x 5-seed")
    print("campaign (gpu_replication_runbook.md, Part A1). NOTE: 30-subset gaps are")
    print("NOT directional -- only the full campaign decides inline-vs-LogIX.")
else:
    print("STOP: LDS near noise. Report back. Adjust the SETUP (epochs / task / n_train),")
    print("NOT the method.")

## Cell 6 — Persist results to Drive + download zip (survive disconnects)

In [ ]:
import os, glob, shutil
dst = "/content/drive/MyDrive/traceprop_results/"
os.makedirs(dst, exist_ok=True)
src_dir = "/content/Traceprop/experiments/results"
copied = 0
for f in glob.glob(src_dir + "/exp35_p160_*"):
    shutil.copy(f, dst); copied += 1
print(f"copied {copied} pilot file(s) to {dst}")

zip_path = shutil.make_archive("/content/p160_pilot_results", "zip", src_dir)
print("zip:", zip_path)
from google.colab import files
files.download(zip_path)

## Cell 7 — PART B (SEPARATE): Pythia-1B in-training overhead

Not part of the pilot. Run this on its own when you want the overhead number
(total = factored logging + inline covariance vs plain training; plus covariance-only
marginal). Uses `torch.cuda.synchronize()` and drops a warmup repeat internally.

In [ ]:
# Cell 7 — PART B (SEPARATE): Pythia-1B in-training overhead + LogIX head-to-head
import subprocess, sys, os, json
os.chdir('/content/Traceprop/experiments')
p = subprocess.run([sys.executable, "exp35_inlineprecond_overhead_hf.py",
                    "--model", "EleutherAI/pythia-1b", "--device", "cuda",
                    "--track", "0", "--kfac", "8", "--steps", "200",
                    "--batch", "8", "--seq", "128", "--repeats", "10", "--with_logix",
                    "--out", "results/exp35_inlineprecond_overhead_hf_pythia1b.json"],
                   capture_output=True, text=True)
os.makedirs("/content/drive/MyDrive/traceprop_results/", exist_ok=True)
open("/content/drive/MyDrive/traceprop_results/overhead_stderr.txt","w").write(p.stderr)
if p.returncode != 0:
    print("EXIT", p.returncode, "STDERR tail:\n" + "\n".join(p.stderr.splitlines()[-40:]))
    raise SystemExit("overhead run failed")
print("\n".join(p.stdout.splitlines()[-30:]))
import shutil
shutil.copy("results/exp35_inlineprecond_overhead_hf_pythia1b.json",
            "/content/drive/MyDrive/traceprop_results/")
print("overhead result saved to Drive")